# Unified Amputee MAT Exercise Classifier

This notebook uses only the original Amputee MAT recordings. Excel exports are not loaded.

The available 22-sensor CyberGlove signal supports:

- Exercise B: 17 hand and wrist actions
- Exercise C: 23 grasps and functional actions
- Rest

Exercise D has 9 actions but its MAT files contain no `glove` variable; D must be modeled separately from EMG/force. The current model therefore learns Amputee MAT glove-frequency patterns for B/C and predicts the named exercise/subexercise label. This is a kinematic-sensor experiment, not EMG-only prosthetic control.

In [1]:
from pathlib import Path
from collections import Counter
import json, random, re
import numpy as np
import torch
from scipy.io import loadmat
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path.cwd()
if not (ROOT/'Database').exists(): ROOT=ROOT.parent
DATA_ROOT=ROOT/'Database'
WINDOW_SAMPLES=400; STRIDE_SAMPLES=100; FFT_BINS=32; CHANNELS=22; MIN_PURITY=.80; CLASS_COUNT=62
ACTIONS={
'A':['Index flexion','Index extension','Middle flexion','Middle extension','Ring flexion','Ring extension','Little finger flexion','Little finger extension','Thumb adduction','Thumb abduction','Thumb flexion','Thumb extension'],
'B':['Thumb up','Extension of index and middle flexion of the others','Flexion of ring and little finger, extension of the others','Thumb opposing base of little finger','Abduction of all fingers','Fingers flexed together in fist','Pointing index','Adduction of extended fingers','Wrist supination (axis: middle finger)','Wrist pronation (axis: middle finger)','Wrist supination (axis: little finger)','Wrist pronation (axis: little finger)','Wrist flexion','Wrist extension','Wrist radial deviation','Wrist ulnar deviation','Wrist extension with closed hand'],
'C':['Large diameter grasp','Small diameter grasp (power grip)','Fixed hook grasp','Index finger extension grasp','Medium wrap','Ring grasp','Prismatic four fingers grasp','Stick grasp','Writing tripod grasp','Power sphere grasp','Three finger sphere grasp','Precision sphere grasp','Tripod grasp','Prismatic pinch grasp','Tip pinch grasp','Quadpod grasp','Lateral grasp','Parallel extension grasp','Extension type grasp','Power disk grasp','Open a bottle with a tripod grasp','Turn a screw (grasp the screwdriver with a stick grasp)','Cut something (grasp the knife with an index finger extension grasp)'],
'D':['Flexion of the little finger','Flexion of the ring finger','Flexion of the middle finger','Flexion of the index finger','Abduction of the thumb','Flexion of the thumb','Flexion of index and little finger','Flexion of ring and middle finger','Flexion of index finger and thumb']}
EXERCISE_NAMES={'A':'Finger basics','B':'Hand configurations and wrist','C':'Grasps and functional movements','D':'Combined finger actions','R':'Rest'}
print('device:',DEVICE,'classes:',1+sum(len(v) for v in ACTIONS.values()))


device: cpu classes: 62


## MAT-only file-to-taxonomy mapping and sensor availability

This canonical classifier uses only the original Amputee MAT recordings. Excel exports are intentionally excluded from the analysis.

- Amputee E1 -> Exercise B: 17 actions plus Rest
- Amputee E2 -> Exercise C: 23 actions plus Rest; raw labels 18-40 are remapped to local labels 1-23
- Amputee E3 -> Exercise D: 9 actions, but these files contain no `glove` variable, so they cannot enter this 22-sensor frequency experiment
- Label 0 -> Rest

The current 22-sensor run therefore contains Exercise B and C plus Rest. Exercise D requires a separate EMG/force experiment because its MAT files do not contain the 22 glove channels.

In [2]:
def taxonomy_for_file(path, database_exercise):
    cohort = 'Amputee' if 'Amputee' in str(path) else 'Intact'
    if cohort == 'Intact':
        mapping = {1: 'A', 2: 'B', 3: 'C'}
    else:
        mapping = {1: 'B', 2: 'C', 3: 'D'}
    return mapping[database_exercise]


def local_action_label(exercise, raw_label, cohort):
    raw_label = int(raw_label)
    if raw_label == 0:
        return 0
    if cohort == 'Amputee' and exercise == 'C':
        return raw_label - 17
    if cohort == 'Amputee' and exercise == 'D':
        return raw_label - 40
    return raw_label


def action_name(exercise, label):
    if int(label) == 0:
        return 'Rest'
    return ACTIONS[exercise][int(label) - 1]


def load_mat_recording(path):
    data = loadmat(path, simplify_cells=True)
    required = ('glove', 'restimulus', 'rerepetition', 'subject', 'exercise')
    missing = [key for key in required if key not in data]
    if missing:
        raise ValueError(f'{path.name} has no 22-sensor glove input; missing {missing}')
    return {key: np.asarray(data[key]) for key in required}


def load_xlsx_recording(path):
    from openpyxl import load_workbook
    workbook = load_workbook(path, read_only=True, data_only=True)
    if 'glove' not in workbook.sheetnames:
        raise ValueError(f'{path.name} has no glove worksheet')
    def array(sheet):
        rows = list(workbook[sheet].iter_rows(min_row=2, values_only=True))
        return np.asarray([[float(value) for value in row[1:]] for row in rows], dtype=np.float32)
    metadata = {str(row[0]): row[1] for row in workbook['metadata'].iter_rows(min_row=2, values_only=True) if row[0] is not None}
    result = {'glove': array('glove'), 'restimulus': array('restimulus')[:, 0], 'rerepetition': array('rerepetition')[:, 0], 'subject': np.array([metadata['subject']]), 'exercise': np.array([metadata['exercise']])}
    workbook.close()
    return result

In [3]:
def make_frequency_windows(recording, path):
    glove = np.asarray(recording['glove'], dtype=np.float32)
    raw_labels = np.asarray(recording['restimulus']).reshape(-1).astype(int)
    groups = np.asarray(recording['rerepetition']).reshape(-1).astype(int)
    database_exercise = int(np.asarray(recording['exercise']).reshape(-1)[0])
    exercise = taxonomy_for_file(path, database_exercise)
    subject = f"Amputee:S{int(np.asarray(recording['subject']).reshape(-1)[0])}"
    if glove.shape[1] != CHANNELS:
        raise ValueError(f'{path.name} expected 22 glove channels, got {glove.shape[1]}')
    labels = np.asarray([local_action_label(exercise, value, 'Amputee') for value in raw_labels])
    if np.any(labels < 0) or np.any(labels > len(ACTIONS[exercise])):
        raise ValueError(f'{path.name} labels do not fit {exercise} taxonomy')
    values, targets, group_ids, names = [], [], [], []
    for start in range(0, len(glove) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
        stop = start + WINDOW_SAMPLES
        block = labels[start:stop]
        rep = groups[start:stop]
        counts = np.bincount(block, minlength=50)
        label = int(counts.argmax())
        if counts[label] / WINDOW_SAMPLES < MIN_PURITY or np.any(rep != rep[0]):
            continue
        signal = glove[start:stop].astype(np.float32)
        signal = signal - np.mean(signal, axis=0, keepdims=True)
        spectrum = np.abs(np.fft.rfft(signal, axis=0))[:FFT_BINS]
        spectrum = np.log1p(spectrum).T.reshape(-1)
        target = f'{exercise}:{label}'
        values.append(spectrum)
        targets.append(target)
        group_ids.append(f'{subject}:{exercise}:rep{int(rep[WINDOW_SAMPLES // 2])}')
        names.append(action_name(exercise, label))
    return np.asarray(values, dtype=np.float32), np.asarray(targets), np.asarray(group_ids), np.asarray(names)

paths = sorted((DATA_ROOT / 'Amputee').glob('*_E*_A1.mat'))
records = []
for path in paths:
    try:
        records.append(make_frequency_windows(load_mat_recording(path), path))
    except Exception as error:
        print('skipped', path.name, error)
X = np.concatenate([item[0] for item in records])
target_names = np.concatenate([item[1] for item in records])
groups = np.concatenate([item[2] for item in records])
names = np.concatenate([item[3] for item in records])
classes = sorted(np.unique(target_names))
class_to_id = {name: index for index, name in enumerate(classes)}
y = np.asarray([class_to_id[name] for name in target_names])
print('Amputee MAT-only glove windows:', X.shape, 'classes:', len(classes), 'exercises:', sorted({name.split(':')[0] for name in classes}))

skipped S10_E3_A1.mat S10_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S11_E3_A1.mat S11_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S1_E3_A1.mat S1_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S2_E3_A1.mat S2_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S3_E3_A1.mat S3_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S4_E3_A1.mat S4_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S5_E3_A1.mat S5_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S6_E3_A1.mat S6_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S7_E3_A1.mat S7_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S8_E3_A1.mat S8_E3_A1.mat has no 22-sensor glove input; missing ['glove']
skipped S9_E3_A1.mat S9_E3_A1.mat has no 22-sensor glove input; missing ['glove']
Amputee MAT-only glove windows: (443849, 704) classes: 42 exercises: ['B', 'C']


In [6]:
subjects = np.asarray([group.split(':', 1)[0] for group in groups])
rng = np.random.default_rng(SEED)
train_groups, test_groups = set(), set()
for subject in np.unique(subjects):
    values = np.unique(groups[subjects == subject])
    rng.shuffle(values)
    split = max(1, round(len(values) * 0.2))
    test_groups.update(values[:split])
    train_groups.update(values[split:])
train = np.isin(groups, list(train_groups))
test = np.isin(groups, list(test_groups))
selected = []
for label in np.unique(y[train]):
    indices = np.flatnonzero(train & (y == label))
    selected.extend(rng.choice(indices, min(1000, len(indices)), replace=False))
selected = np.asarray(selected)
rng.shuffle(selected)
X_train, y_train = X[selected], y[selected]
X_test, y_test = X[test], y[test]
exercise_names = np.asarray([classes[index].split(':')[0] for index in y])
exercise_ids = sorted(np.unique(exercise_names))
exercise_to_id = {name: index for index, name in enumerate(exercise_ids)}
y_exercise = np.asarray([exercise_to_id[name] for name in exercise_names])

class FrequencyMLP(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(X.shape[1], 512), nn.GELU(), nn.Dropout(.05), nn.Linear(512, 256), nn.GELU(), nn.Linear(256, n_classes))
    def forward(self, values):
        return self.net(values)

def fit_frequency_model(features, labels, n_classes, epochs=200):
    fitted = FrequencyMLP(n_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(fitted.parameters(), lr=1e-3, weight_decay=1e-5)
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels).long())
    loader = DataLoader(dataset, batch_size=512, shuffle=True)
    for epoch in range(epochs):
        fitted.train()
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            nn.functional.cross_entropy(fitted(batch_x), batch_y).backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            print('model epochs', epoch + 1, 'classes', n_classes)
    return fitted

exercise_model = fit_frequency_model(X_train, y_exercise[selected], len(exercise_ids))
action_models = {}
for exercise in exercise_ids:
    global_ids = [index for index, name in enumerate(classes) if name.startswith(f'{exercise}:')]
    local_map = {global_id: local for local, global_id in enumerate(global_ids)}
    keep = np.asarray([classes[label].startswith(f'{exercise}:') for label in y_train])
    local_labels = np.asarray([local_map[label] for label in y_train[keep]])
    action_models[exercise] = (fit_frequency_model(X_train[keep], local_labels, len(global_ids)), global_ids)

model epochs 10 classes 2
model epochs 20 classes 2
model epochs 30 classes 2
model epochs 40 classes 2
model epochs 50 classes 2
model epochs 60 classes 2
model epochs 70 classes 2
model epochs 80 classes 2
model epochs 90 classes 2
model epochs 100 classes 2
model epochs 110 classes 2
model epochs 120 classes 2
model epochs 130 classes 2
model epochs 140 classes 2
model epochs 150 classes 2
model epochs 160 classes 2
model epochs 170 classes 2
model epochs 180 classes 2
model epochs 190 classes 2
model epochs 200 classes 2
model epochs 10 classes 18
model epochs 20 classes 18
model epochs 30 classes 18
model epochs 40 classes 18
model epochs 50 classes 18
model epochs 60 classes 18
model epochs 70 classes 18
model epochs 80 classes 18
model epochs 90 classes 18
model epochs 100 classes 18
model epochs 110 classes 18
model epochs 120 classes 18
model epochs 130 classes 18
model epochs 140 classes 18
model epochs 150 classes 18
model epochs 160 classes 18
model epochs 170 classes 18
mo

In [7]:
exercise_model.eval()
with torch.no_grad():
    predicted_exercises = exercise_model(torch.from_numpy(X_test).to(DEVICE)).argmax(1).cpu().numpy()
true_exercises = y_exercise[test]
predictions = np.empty(len(y_test), dtype=int)
for exercise_index, exercise in enumerate(exercise_ids):
    keep = predicted_exercises == exercise_index
    if not keep.any():
        continue
    action_model, global_ids = action_models[exercise]
    action_model.eval()
    with torch.no_grad():
        local_predictions = action_model(torch.from_numpy(X_test[keep]).to(DEVICE)).argmax(1).cpu().numpy()
    predictions[keep] = np.asarray(global_ids)[local_predictions]
print({'exercise_accuracy': accuracy_score(true_exercises, predicted_exercises), 'accuracy': accuracy_score(y_test, predictions), 'balanced_accuracy': balanced_accuracy_score(y_test, predictions), 'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0), 'classes': len(classes), 'test_windows': len(y_test)})
print(classification_report(y_test, predictions, target_names=classes, zero_division=0))

{'exercise_accuracy': 0.8292329472616407, 'accuracy': 0.10705964974862608, 'balanced_accuracy': 0.11118234842784361, 'macro_f1': 0.09559268905167247, 'classes': 42, 'test_windows': 89707}
              precision    recall  f1-score   support

         B:0       0.52      0.17      0.26     12413
         B:1       0.31      0.45      0.37      1446
        B:10       0.21      0.11      0.15      1434
        B:11       0.14      0.03      0.05      1470
        B:12       0.01      0.05      0.01      1397
        B:13       0.03      0.06      0.04      1466
        B:14       0.22      0.09      0.13      1383
        B:15       0.20      0.07      0.10      1542
        B:16       0.04      0.13      0.06      1498
        B:17       0.07      0.25      0.11      1315
         B:2       0.11      0.10      0.10      1469
         B:3       0.10      0.09      0.09      1324
         B:4       0.18      0.09      0.12      1422
         B:5       0.10      0.07      0.08      1388
 

## Interpretation

This notebook predicts the named exercise and subexercise from frequency patterns in the 22 glove sensors. Because the glove directly measures hand kinematics, this is not evidence that EMG alone can control a prosthesis. The exercise-specific label mapping is explicit, and the split is by complete subject-repetition groups to prevent overlapping-window leakage.

## Complete 49-movement EMG classifier

The glove-frequency model above covers B/C only because Exercise D has no glove variable. The complete Amputee action task uses the 12 EMG channels and includes B, C, D, plus Rest. Raw label normalization is:

```text
E1: 0..17  -> B: Rest + 17 actions
E2: 0,18..40 -> C: Rest + 23 actions, mapped to local 0..23
E3: 0,41..49 -> D: Rest + 9 actions, mapped to local 0..9
```

This is the model to use when the input is an EMG signal and the requested output is any of the 49 movements. The 22-channel glove model should not be used for D.
